# Preprocessing & Leakage-Free Feature Engineering — Olist Delivery Time Prediction

## Quy ước thời điểm dự đoán: **Purchase-time Prediction**
- Mô hình dự đoán **ngay tại thời điểm khách hàng đặt hàng** (`order_purchase_timestamp`).
- **Loại bỏ hoàn toàn** các trường phát sinh sau thời điểm đặt hàng: `order_approved_at`, `shipping_limit_date`, `order_delivered_carrier_date`, `order_delivered_customer_date`, và `reviews`.
- Áp dụng **Chronological Expanding-Window Target Encoding** để không rò rỉ thông tin tương lai.
- Sửa **Aggregation với `min_count=1`**, bổ sung `missing_weight_count`, `missing_dimension_count`, export **`model/te_maps.json`** cho inference, và xuất thêm bộ dữ liệu **Right-Censoring Cutoff (`<= 2018-06-30`)**.

In [1]:
import json
import os
import numpy as np
import pandas as pd

RAW_DIR = 'raw'
OUT_DIR = 'processed'
MODEL_DIR = 'model'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Các hàm tiện ích (Haversine, Chronological Target Encoding & Split Pipeline)

In [2]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Compute great-circle distance in kilometers between two lat/lon points."""
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))


def compute_smoothed_map(hist_series, hist_target, smoothing=15.0):
    """Return (mapping_dict, global_mean) fitted strictly on historical data."""
    global_mean = float(hist_target.mean())
    stats = hist_target.groupby(hist_series).agg(['count', 'mean'])
    smooth = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    return smooth.to_dict(), global_mean


def add_chronological_target_encoding(train_df, val_df, test_df, col, target_col, ts_col='order_purchase_timestamp', smoothing=15.0, n_blocks=6):
    """Expanding-window chronological target encoding on sorted train_df."""
    feat_name = f'{col}_te'
    n = len(train_df)
    block_edges = np.linspace(0, n, n_blocks + 1, dtype=int)
    train_encoded = pd.Series(index=train_df.index, dtype=float)

    b0_end = block_edges[1]
    b0_df = train_df.iloc[:b0_end]
    prior_mean = float(b0_df[target_col].iloc[0])
    cum_sum = b0_df[target_col].cumsum().shift(1, fill_value=0.0)
    cum_cnt = pd.Series(np.arange(len(b0_df)), index=b0_df.index, dtype=float)
    train_encoded.iloc[:b0_end] = (cum_sum + smoothing * prior_mean) / (cum_cnt + smoothing)

    for k in range(1, n_blocks):
        start_idx, end_idx = block_edges[k], block_edges[k + 1]
        hist_part = train_df.iloc[:start_idx]
        curr_part = train_df.iloc[start_idx:end_idx]
        assert hist_part[ts_col].max() <= curr_part[ts_col].min(), f'Temporal leak detected in block {k}!'
        m_dict, g_mean = compute_smoothed_map(hist_part[col], hist_part[target_col], smoothing=smoothing)
        train_encoded.iloc[start_idx:end_idx] = curr_part[col].map(m_dict).fillna(g_mean)

    full_dict, full_g_mean = compute_smoothed_map(train_df[col], train_df[target_col], smoothing=smoothing)
    train_df[feat_name] = train_encoded
    val_df[feat_name] = val_df[col].map(full_dict).fillna(full_g_mean)
    test_df[feat_name] = test_df[col].map(full_dict).fillna(full_g_mean)
    return feat_name, full_dict, full_g_mean

## 2. Đọc & Tổng hợp dữ liệu (Data Integration với `min_count=1`)

In [3]:
orders = pd.read_csv(os.path.join(RAW_DIR, 'olist_orders_dataset.csv'))
order_items = pd.read_csv(os.path.join(RAW_DIR, 'olist_order_items_dataset.csv'))
customers = pd.read_csv(os.path.join(RAW_DIR, 'olist_customers_dataset.csv'))
sellers = pd.read_csv(os.path.join(RAW_DIR, 'olist_sellers_dataset.csv'))
products = pd.read_csv(os.path.join(RAW_DIR, 'olist_products_dataset.csv'))
geolocation = pd.read_csv(os.path.join(RAW_DIR, 'olist_geolocation_dataset.csv'))
payments = pd.read_csv(os.path.join(RAW_DIR, 'olist_order_payments_dataset.csv'))

for col in ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col])

df = orders[orders['order_status'] == 'delivered'].copy()
df = df.dropna(subset=['order_delivered_customer_date', 'order_purchase_timestamp']).copy()
df['delivery_time_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400.0
df = df[df['delivery_time_days'] > 0].copy()
df['olist_estimated_days'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400.0

geo_valid = geolocation[
    geolocation['geolocation_lat'].between(-35.0, 6.0)
    & geolocation['geolocation_lng'].between(-75.0, -30.0)
]
geo_zip = geo_valid.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].median().reset_index()
geo_state = geo_valid.groupby('geolocation_state')[['geolocation_lat', 'geolocation_lng']].median().reset_index()

products_clean = products.copy()
for dim_col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    invalid_cnt = int((products_clean[dim_col] <= 0).sum())
    if invalid_cnt > 0:
        print(f'Setting {invalid_cnt} non-positive values in {dim_col} to NaN')
        products_clean.loc[products_clean[dim_col] <= 0, dim_col] = np.nan

items = order_items.merge(products_clean, on='product_id', how='left', validate='many_to_one')
items = items.merge(sellers, on='seller_id', how='left', validate='many_to_one')
items['product_volume_cm3'] = items['product_length_cm'] * items['product_height_cm'] * items['product_width_cm']
items['is_missing_weight'] = items['product_weight_g'].isna().astype(int)
items['is_missing_dim'] = items['product_volume_cm3'].isna().astype(int)

order_agg = (
    items.groupby('order_id')
    .agg(
        total_items=('order_item_id', 'count'),
        total_price=('price', 'sum'),
        total_freight=('freight_value', 'sum'),
        num_unique_sellers=('seller_id', 'nunique'),
        num_unique_products=('product_id', 'nunique'),
        total_weight_g=('product_weight_g', lambda s: s.sum(min_count=1)),
        max_weight_g=('product_weight_g', 'max'),
        total_volume_cm3=('product_volume_cm3', lambda s: s.sum(min_count=1)),
        missing_weight_count=('is_missing_weight', 'sum'),
        missing_dimension_count=('is_missing_dim', 'sum'),
        avg_photos_qty=('product_photos_qty', 'mean'),
        primary_seller_id=('seller_id', 'first'),
        primary_seller_zip=('seller_zip_code_prefix', 'first'),
        primary_seller_city=('seller_city', 'first'),
        primary_seller_state=('seller_state', 'first'),
        primary_category=('product_category_name', 'first'),
    )
    .reset_index()
)

pay_agg = (
    payments.groupby('order_id')
    .agg(
        primary_payment_type=('payment_type', 'first'),
        max_installments=('payment_installments', 'max'),
        total_payment_value=('payment_value', 'sum'),
        num_payment_methods=('payment_sequential', 'count'),
    )
    .reset_index()
)

df = df.merge(customers, on='customer_id', how='left', validate='one_to_one')
df = df.merge(order_agg, on='order_id', how='left', validate='one_to_one')
df = df.merge(pay_agg, on='order_id', how='left', validate='one_to_one')
assert df['order_id'].is_unique, 'Duplicate order_id after merge!'

df = df.merge(
    geo_zip.rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix', 'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lng'}),
    on='customer_zip_code_prefix', how='left'
)
df = df.merge(
    geo_zip.rename(columns={'geolocation_zip_code_prefix': 'primary_seller_zip', 'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lng'}),
    on='primary_seller_zip', how='left'
)

state_lat_map = geo_state.set_index('geolocation_state')['geolocation_lat']
state_lng_map = geo_state.set_index('geolocation_state')['geolocation_lng']
df['cust_lat'] = df['cust_lat'].fillna(df['customer_state'].map(state_lat_map))
df['cust_lng'] = df['cust_lng'].fillna(df['customer_state'].map(state_lng_map))
df['sell_lat'] = df['sell_lat'].fillna(df['primary_seller_state'].map(state_lat_map))
df['sell_lng'] = df['sell_lng'].fillna(df['primary_seller_state'].map(state_lng_map))
print(f'Merged shape: {df.shape}')

Setting 4 non-positive values in product_weight_g to NaN
Merged shape: (96470, 38)


## 3. Tạo đặc trưng hợp lệ tại thời điểm đặt hàng (Purchase-time Features)

In [4]:
df['haversine_km'] = haversine_km(df['sell_lat'], df['sell_lng'], df['cust_lat'], df['cust_lng'])
df['is_same_state'] = (df['customer_state'] == df['primary_seller_state']).astype(int)
df['is_same_city'] = (df['customer_city'].astype(str).str.lower() == df['primary_seller_city'].astype(str).str.lower()).astype(int)
df['route'] = df['primary_seller_state'].fillna('UNK') + '_' + df['customer_state'].fillna('UNK')

ts = df['order_purchase_timestamp']
df['purchase_hour'] = ts.dt.hour
df['purchase_dayofweek'] = ts.dt.day_of_week
df['purchase_day'] = ts.dt.day
df['purchase_month'] = ts.dt.month
df['is_weekend'] = (df['purchase_dayofweek'] >= 5).astype(int)
df['hour_sin'] = np.sin(2 * np.pi * df['purchase_hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['purchase_hour'] / 24.0)
df['month_sin'] = np.sin(2 * np.pi * df['purchase_month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['purchase_month'] / 12.0)

df['freight_ratio'] = df['total_freight'] / (df['total_price'] + 1e-5)
df['price_per_item'] = df['total_price'] / df['total_items'].clip(lower=1)

df['primary_category'] = df['primary_category'].fillna('unknown')
df['primary_payment_type'] = df['primary_payment_type'].fillna('unknown')
df['customer_state'] = df['customer_state'].fillna('UNK')
df['primary_seller_state'] = df['primary_seller_state'].fillna('UNK')
print('Purchase-time feature engineering completed!')

Purchase-time feature engineering completed!


## 4. Chia tập theo thời gian, Export TE Maps (`te_maps.json`) & Xuất bản Full + Cutoff (`<= 2018-06-30`)

In [5]:
base_num_features = [
    'haversine_km', 'cust_lat', 'cust_lng', 'sell_lat', 'sell_lng',
    'is_same_state', 'is_same_city', 'purchase_hour', 'purchase_dayofweek',
    'purchase_day', 'purchase_month', 'is_weekend', 'hour_sin', 'hour_cos',
    'month_sin', 'month_cos', 'total_items', 'total_price', 'total_freight',
    'num_unique_sellers', 'num_unique_products', 'total_weight_g', 'max_weight_g',
    'total_volume_cm3', 'missing_weight_count', 'missing_dimension_count',
    'avg_photos_qty', 'freight_ratio', 'price_per_item',
    'max_installments', 'total_payment_value', 'num_payment_methods',
]
cat_features = ['customer_state', 'primary_seller_state', 'primary_payment_type', 'primary_category']
te_cols_src = ['primary_seller_id', 'route', 'customer_state', 'primary_category']

def prepare_and_save_split(source_df, prefix=''):
    s_df = source_df.sort_values('order_purchase_timestamp').reset_index(drop=True)
    n = len(s_df)
    tr_end, va_end = int(0.70 * n), int(0.85 * n)
    tr_df = s_df.iloc[:tr_end].copy()
    va_df = s_df.iloc[tr_end:va_end].copy()
    te_df = s_df.iloc[va_end:].copy()

    te_names, te_maps = [], {}
    for col in te_cols_src:
        te_name, m_dict, g_mean = add_chronological_target_encoding(tr_df, va_df, te_df, col=col, target_col='delivery_time_days', smoothing=15.0, n_blocks=6)
        te_names.append(te_name)
        te_maps[col] = {str(k): float(v) for k, v in m_dict.items()}
        te_maps[f'{col}__global_mean'] = float(g_mean)

    num_cols = base_num_features + te_names
    for col in num_cols:
        med = tr_df[col].median()
        tr_df[col] = tr_df[col].fillna(med)
        va_df[col] = va_df[col].fillna(med)
        te_df[col] = te_df[col].fillna(med)

    for part in (tr_df, va_df, te_df):
        part['freight_per_kg'] = part['total_freight'] / ((part['total_weight_g'] / 1000.0) + 0.1)
        part['volumetric_weight_g'] = part['total_volume_cm3'] / 6.0

    final_num = num_cols + ['freight_per_kg', 'volumetric_weight_g']
    keep_cols = ['order_id', 'order_purchase_timestamp', 'delivery_time_days', 'olist_estimated_days'] + final_num + cat_features
    tr_df[keep_cols].to_csv(os.path.join(OUT_DIR, f'{prefix}train.csv'), index=False)
    va_df[keep_cols].to_csv(os.path.join(OUT_DIR, f'{prefix}val.csv'), index=False)
    te_df[keep_cols].to_csv(os.path.join(OUT_DIR, f'{prefix}test.csv'), index=False)
    print(f'[{prefix or "full"}] Train={len(tr_df):,} | Val={len(va_df):,} | Test={len(te_df):,} ({te_df["order_purchase_timestamp"].min().date()} -> {te_df["order_purchase_timestamp"].max().date()})')
    return final_num, te_maps, {'train': len(tr_df), 'val': len(va_df), 'test': len(te_df)}

# 1. Phiên bản Full
final_num, te_maps, split_sizes = prepare_and_save_split(df, prefix='')
with open(os.path.join(MODEL_DIR, 'te_maps.json'), 'w', encoding='utf-8') as f:
    json.dump(te_maps, f, indent=2, ensure_ascii=False)

metadata = {
    'prediction_time': 'purchase_time',
    'target_col': 'delivery_time_days',
    'benchmark_col': 'olist_estimated_days',
    'num_features': final_num,
    'cat_features': cat_features,
    'split_sizes': split_sizes,
}
with open(os.path.join(OUT_DIR, 'feature_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# 2. Phiên bản Cutoff <= 2018-06-30 (P0.2 kiểm tra right-censoring)
df_cutoff = df[df['order_purchase_timestamp'] <= pd.Timestamp('2018-06-30 23:59:59')].copy()
prepare_and_save_split(df_cutoff, prefix='cutoff_')
print('Saved processed datasets, cutoff datasets, metadata & model/te_maps.json!')

[full] Train=67,529 | Val=14,470 | Test=14,471 (2018-06-21 -> 2018-08-29)
[cutoff_] Train=58,774 | Val=12,594 | Test=12,595 (2018-05-01 -> 2018-06-30)
Saved processed datasets, cutoff datasets, metadata & model/te_maps.json!
